# TISER Baseline — Colab Runner

Install → fetch data → train → evaluate the TISER QLoRA baseline.

**Compute note:** the committed `config.yaml` defaults to the full 60k-example training run, which needs an **A100** (Runtime → Change runtime type → A100). On a **free-tier T4** the full run will not finish — use the subset knobs in the train/eval cells below for a smoke run.

In [ ]:
!nvidia-smi

In [ ]:
# If running fresh on Colab, clone the repo. Skip if already opened from it.
# !git clone <YOUR_REPO_URL> tiser_baseline
# %cd tiser_baseline

In [ ]:
# Use Colab's bundled torch (do NOT reinstall it — it breaks bitsandbytes).
!pip install -r requirements.txt

In [ ]:
!pip install -e .

In [ ]:
# Optional: persist adapters/outputs to Drive across sessions.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# On Colab the local POLITO path is absent, so this falls back to the HF mirror / git-lfs.
!python scripts/fetch_data.py --config config/config.yaml

## Train
On a **T4**, keep `--subset` small (e.g. 2000) for a smoke run.
On an **A100**, drop `--subset` to train on the full set and optionally raise `train.per_device_batch_size` in `config.yaml`.

In [ ]:
!python scripts/train.py --config config/config.yaml --subset 2000

In [ ]:
# T4: sample each split. A100: drop the flag to evaluate the full test set.
!python scripts/evaluate.py --config config/config.yaml --max-samples-per-split 200

In [ ]:
import json, yaml

run_name = yaml.safe_load(open('config/config.yaml'))['run_name']
metrics = json.load(open(f'outputs/{run_name}/metrics.json'))

print(f"macro-EM {metrics['macro_em']:.3f} | macro-F1 {metrics['macro_f1']:.3f}")
print(f"malformed: {metrics['n_malformed']}/{metrics['n_total']}\n")
for split, m in metrics['per_split'].items():
    print(f"{split:16s} EM {m['em']:.3f}  F1 {m['f1']:.3f}  (n={m['n']})")

In [ ]:
# Inspect a few predictions: raw generation, parsed answer, gold, EM/F1.
with open(f'outputs/{run_name}/predictions.jsonl') as f:
    for line in list(f)[:3]:
        r = json.loads(line)
        print(f"[{r['dataset_name']}] gold={r['gold']!r} pred={r['pred_answer']!r} "
              f"em={r['em']} f1={r['f1']:.2f} malformed={r['malformed']}")
        print('  raw:', r['raw_generation'][:300].replace('\n', ' '), '\n')

## Scaling to the full reproduction (A100)
Set `train.subset_size: null` and `eval.max_samples_per_split: null` in `config.yaml` (or just omit the CLI flags above), and raise `train.per_device_batch_size` (4–8) and `eval.batch_size` (16–32). No code changes needed.

Target (paper Table 1, Qwen2.5-7B + TISER): **91.1 macro-EM / 94.4 macro-F1**.